# Training Data Quality and Lineage: Stop the Bad Run Before It Starts

> **The story:** In 1996, computer scientist Hal Abelson argued that programs must be written for people to read and only incidentally for machines to execute. Modern dataset lineage applies the same idea to training data: a row is not release evidence unless another engineer can inspect where it came from, why it belongs, and which checks admitted it. Riverside House is about to learn that a parseable file is not the same thing as a defensible training release.
>
> **Where you are:** The [fine-tuning data notebook](../../../genai/02-llm-finetuning/01-llm-finetuning-data-techniques.ipynb) taught Riverside how continued pretraining, SFT, and preference pairs change behavior. This chapter moves one gate earlier: before an objective or optimizer sees a row, you must prove that the dataset is isolated, valid, authorized, representative, and reproducible.
>
> **Notation:** $D$ is the ordered dataset; $r_i$ is one row; $S(r_i,r_j)$ is token-set Jaccard similarity; $H(D)$ is a SHA-256 fingerprint; $n_s$ is the count for slice $s$; $G$ is the promotion gate.

> **What you finished last time:** objective selection, held-out evidence, and training manifests in the Riverside fine-tuning arc.
> **What this notebook delivers:** a deterministic issue ledger, source and candidate fingerprints, and promotion reports written to `./artifacts/` when run.
> **Prerequisite for the next notebook:** a promoted dataset fingerprint that later model and release manifests can reference.

| Part | Riverside failure | Evidence computed |
| --- | --- | --- |
| 0 | Thirteen schema-valid rows look ready | A numbered release incident |
| 1 | Parsing is mistaken for fitness | Immutable load, schema report, source digest |
| 2 | Random splitting leaks policy text | Exact and cross-split near-duplicate clusters |
| 3 | Schema-valid messages cannot train a target | Role-order and assistant-target checks |
| 4 | Valid text is unsafe or untraceable | PII, provenance, rights, and contamination flags |
| 5 | Aggregate size hides coverage gaps | Task, split, and slice distributions |
| 6 | Preferences teach contradiction and verbosity | Disagreement and shortcut-risk findings |
| 7 | Console findings cannot gate release | Canonical fingerprints and promotion JSON |
| 8 | A pass/fail result loses its limits | Evidence ledger and conceptual handoff |

## 0 - The Challenge

> **The mission:** Riverside House must prevent an unsafe training-data candidate from reaching a fine-tuning job and retain evidence that explains the decision.

**What we know so far:**

- The fixture contract contains 13 rows: 10 SFT and 3 preference examples.
- All 13 are expected to pass structural JSON Schema validation.
- The declared split is 12 train rows and 1 validation row.
- **But the contract says 12 of 13 rows have at least one known issue; only `td-004` is clean.**

**What's blocking us:**
Riverside's ingestion job reports 100% structural validity. That green number hides one train/validation leakage cluster, two invalid chat templates, two PII findings in one row, missing or unapproved lineage, one eval-reserved source in training, one contradictory preference group, and one length-shortcut risk. Training would turn these defects into behavior and still produce an ordinary loss curve.

**What this chapter unlocks:**
A deterministic pre-training gate that measures every failure from the immutable fixture, fingerprints source and curated candidates, and emits a promotion report. Expected facts in prose are fixture-contract targets; code computes and asserts the results.

```mermaid
flowchart LR
    A["13 JSONL rows"] --> B["Schema validation"]
    B --> C["13 structurally valid"]
    C --> D["Semantic and policy audit"]
    D --> E["12 rows with issues"]
    E --> F["Promotion denied"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Which claim survives the first proof?

1. Schema validation rejects most unsafe rows.
2. All rows pass schema, so semantic and policy gates own the block.
3. Malformed rows prevent deterministic hashing.

The next code cell resolves it.

## 1 - Immutable Load and Structural Validity

A notebook that silently reads a copied file cannot prove which candidate it audited. Hash raw bytes before parsing, then validate every JSONL object against the committed schema. Structural validity is the first gate, not the promotion decision.

```mermaid
flowchart LR
    A["Shared fixture bytes"] --> B["SHA-256 source digest"]
    A --> C["JSONL parser"]
    C --> D["Draft 2020-12 validator"]
    D --> E{"Structural errors?"}
    E -->|"yes"| F["Block ingestion"]
    E -->|"no"| G["Continue to semantic gates"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Load immutable shared fixtures -------------------------------------------
from __future__ import annotations

from collections import Counter, defaultdict
from copy import deepcopy
from hashlib import sha256
from itertools import combinations
from pathlib import Path
import json
import re
import unicodedata

from jsonschema import Draft202012Validator


def find_fixture_dir(start: Path) -> Path:
    candidates = []
    for base in (start, *start.parents):
        candidates.extend([
            base / "learning" / "ai-engineer" / "shared" / "training-data",
            base / "shared" / "training-data",
            base.parent / "shared" / "training-data",
        ])
    for candidate in candidates:
        resolved = candidate.resolve()
        if (resolved / "training-examples.jsonl").is_file():
            return resolved
    raise FileNotFoundError("Could not locate learning/role-based-tracks/ai-engineer/shared/training-data")


def verify_fixture_contract(fixture_dir: Path, relative_paths: tuple[str, ...]) -> str:
    shared_dir = fixture_dir.parent
    version = (shared_dir / "VERSION").read_text(encoding="utf-8").strip()
    manifest = json.loads((shared_dir / "fixture-manifest.json").read_text(encoding="utf-8"))
    if manifest["fixture_version"] != version:
        raise RuntimeError("Fixture VERSION and fixture-manifest.json disagree")
    for relative_path in relative_paths:
        expected = manifest["files"].get(relative_path)
        if expected is None:
            raise RuntimeError(f"Fixture manifest does not pin {relative_path}")
        actual = sha256((shared_dir / relative_path).read_bytes()).hexdigest()
        if actual != expected:
            raise RuntimeError(
                f"Stale or modified fixture: {relative_path}. "
                "Restore the pinned bytes or intentionally version the shared fixture contract."
            )
    return version


FIXTURE_DIR = find_fixture_dir(Path.cwd())
DATA_PATH = FIXTURE_DIR / "training-examples.jsonl"
SCHEMA_PATH = FIXTURE_DIR / "training-example.schema.json"
FIXTURE_VERSION = verify_fixture_contract(
    FIXTURE_DIR,
    (
        "training-data/training-examples.jsonl",
        "training-data/training-example.schema.json",
        "training-data/EXPECTED_OUTCOMES.md",
    ),
)
SOURCE_BYTES = DATA_PATH.read_bytes()
SOURCE_FILE_SHA256 = sha256(SOURCE_BYTES).hexdigest()
ROWS = [json.loads(line) for line in SOURCE_BYTES.decode("utf-8").splitlines() if line.strip()]
SCHEMA = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
ROWS_BY_ID = {row["example_id"]: row for row in ROWS}

validator = Draft202012Validator(SCHEMA)
schema_errors = {}
for row in ROWS:
    errors = sorted(validator.iter_errors(row), key=lambda error: list(error.path))
    if errors:
        schema_errors[row["example_id"]] = [error.message for error in errors]
schema_valid_count = len(ROWS) - len(schema_errors)

assert len(ROWS) == 13, "Fixture contract changed: expected 13 rows"
assert schema_valid_count == 13, f"Expected 13 schema-valid rows, found {schema_valid_count}"
print(f"Verified fixture contract: {FIXTURE_VERSION}")
print(f"Measured from fixture: {len(ROWS)} rows")
print(f"Measured from fixture: source SHA-256 {SOURCE_FILE_SHA256}")
print(f"Measured from fixture: {schema_valid_count}/{len(ROWS)} rows pass JSON Schema")
print("Prediction resolved: structural validity is 100%, while training fitness remains unproven.")

**Common Pitfalls**

| | Pattern | Why it matters |
| --- | --- | --- |
| Wrong | Validate one parsed object and discard source bytes | You cannot prove which file produced the report |
| Right | Hash bytes, parse every line, validate every row | Byte identity and structural evidence remain inspectable |
| Wrong | Treat `13/13 schema valid` as promotion | Schema cannot infer role order, authorization, leakage, or agreement |
| Right | Use structural validity as the first gate | Each failure is owned by an observable check |

**Quick Health Check:** verify paths, unique IDs, one schema version, and validation coverage.

In [ ]:
# -- Health check: fixture identity and structural coverage -----------------
example_ids = [row["example_id"] for row in ROWS]
health_1 = {
    "data_file_exists": DATA_PATH.is_file(),
    "schema_file_exists": SCHEMA_PATH.is_file(),
    "ids_are_unique": len(example_ids) == len(set(example_ids)),
    "schema_version_is_v1": {row["schema_version"] for row in ROWS} == {"ai-eng.training-example.v1"},
    "every_row_schema_validated": schema_valid_count == len(ROWS),
}
assert all(health_1.values()), health_1
print(json.dumps(health_1, indent=2, sort_keys=True))
print("PASS: source identity and structural coverage are reproducible.")

**Reflection bridge:** The parser found no reason to complain. Two rows can each be structurally perfect while making validation meaningless because they repeat the same policy language across a split boundary.

## 2 - Duplicate and Split Leakage

Riverside split rows before grouping related source text. One exact duplicate remains inside training, and a lightly reworded version crosses into validation. Exact hashes catch the first failure; normalized similarity catches the second.

$$S(A,B)=\frac{|A \cap B|}{|A \cup B|}$$

The score is the fraction of unique normalized tokens shared by both rows. It is interpretable and useful for this fixture, not a universal semantic-equivalence claim.

```mermaid
flowchart LR
    A["Canonical learning text"] --> B["Exact SHA-256 groups"]
    A --> C["Normalize and tokenize"]
    C --> D["Pairwise Jaccard score"]
    B --> E["Duplicate edges"]
    D --> F{"Cross-split and above threshold?"}
    F -->|"yes"| G["Leakage edge"]
    E --> H["Connected clusters"]
    G --> H
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Will exact hashing find the train/validation leak, or stop at repeated train rows? The next cell compares both detectors on the same content.

In [ ]:
# -- Measure exact duplicates and cross-split near duplicates ---------------
def learning_text(row: dict) -> str:
    parts = [message["content"] for message in row["messages"]]
    parts.extend(value for value in (row["chosen"], row["rejected"]) if value)
    return " ".join(parts)


def normalized_tokens(row: dict) -> set[str]:
    text = unicodedata.normalize("NFKC", learning_text(row)).casefold()
    return set(re.findall(r"[a-z0-9]+", text))


def exact_payload_digest(row: dict) -> str:
    payload = {key: row[key] for key in ("task_type", "messages", "chosen", "rejected")}
    encoded = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
    return sha256(encoded).hexdigest()


def exact_duplicate_groups(rows: list[dict]) -> list[list[str]]:
    grouped = defaultdict(list)
    for row in rows:
        grouped[exact_payload_digest(row)].append(row["example_id"])
    return sorted(sorted(ids) for ids in grouped.values() if len(ids) > 1)


def jaccard(left: set[str], right: set[str]) -> float:
    union = left | right
    return len(left & right) / len(union) if union else 1.0


def cross_split_near_pairs(rows: list[dict], threshold: float) -> list[dict]:
    pairs = []
    for left, right in combinations(rows, 2):
        if left["split"] == right["split"]:
            continue
        score = jaccard(normalized_tokens(left), normalized_tokens(right))
        if score >= threshold:
            pairs.append({"left_id": left["example_id"], "right_id": right["example_id"], "similarity": round(score, 3)})
    return pairs


def leakage_clusters(rows: list[dict], exact: list[list[str]], near: list[dict]) -> list[list[str]]:
    adjacency = defaultdict(set)
    for group in exact:
        for left, right in combinations(group, 2):
            adjacency[left].add(right)
            adjacency[right].add(left)
    for pair in near:
        adjacency[pair["left_id"]].add(pair["right_id"])
        adjacency[pair["right_id"]].add(pair["left_id"])
    rows_by_id = {row["example_id"]: row for row in rows}
    seen, clusters = set(), []
    for start in sorted(adjacency):
        if start in seen:
            continue
        stack, component = [start], set()
        while stack:
            current = stack.pop()
            if current in component:
                continue
            component.add(current)
            stack.extend(adjacency[current] - component)
        seen.update(component)
        if len({rows_by_id[item]["split"] for item in component}) > 1:
            clusters.append(sorted(component))
    return sorted(clusters)


NEAR_DUPLICATE_THRESHOLD = 0.72
exact_groups = exact_duplicate_groups(ROWS)
near_pairs = cross_split_near_pairs(ROWS, NEAR_DUPLICATE_THRESHOLD)
cross_split_clusters = leakage_clusters(ROWS, exact_groups, near_pairs)
assert exact_groups == [["td-001", "td-002"]]
assert cross_split_clusters == [["td-001", "td-002", "td-003"]]
print(f"Measured exact duplicate groups: {exact_groups}")
print(f"Measured cross-split edges: {near_pairs}")
print(f"Measured leakage clusters: {cross_split_clusters}")
print("Prediction resolved: exact hashing stops inside train; similarity exposes validation leakage.")

**Your turn:** Change one variable, not the data. Lower or raise the threshold and predict whether the known cluster remains connected.

In [ ]:
# -- Your turn: inspect near-duplicate threshold sensitivity ----------------
MY_NEAR_DUPLICATE_THRESHOLD = 0.72  # CHANGE THIS: try 0.60, then 0.90
my_pairs = cross_split_near_pairs(ROWS, MY_NEAR_DUPLICATE_THRESHOLD)
my_clusters = leakage_clusters(ROWS, exact_groups, my_pairs)
print(f"Threshold: {MY_NEAR_DUPLICATE_THRESHOLD:.2f}")
print(f"Cross-split edges: {len(my_pairs)}")
print(f"Leakage clusters: {my_clusters}")
print("PASS: the detector was recomputed without editing fixture labels.")

**Common Pitfalls**

| | Pattern | Why it matters |
| --- | --- | --- |
| Wrong | Deduplicate after random splitting | Related text can already occupy train and evaluation |
| Right | Group by source and duplicate cluster before splitting | Evaluation stays independent |
| Wrong | Call every high lexical score a semantic duplicate | Boilerplate can inflate overlap |
| Right | Retain scores and IDs for review | Threshold decisions remain inspectable |

**Quick Health Check:** verify one exact group, a cluster spanning train and validation, expected IDs, and retained scores.

In [ ]:
# -- Health check: duplicate and leakage evidence ---------------------------
health_2 = {
    "one_exact_group": exact_groups == [["td-001", "td-002"]],
    "one_cross_split_cluster": cross_split_clusters == [["td-001", "td-002", "td-003"]],
    "cluster_spans_splits": {ROWS_BY_ID[item]["split"] for item in cross_split_clusters[0]} == {"train", "validation"},
    "scores_retained": all(0.0 <= pair["similarity"] <= 1.0 for pair in near_pairs),
}
assert all(health_2.values()), health_2
print(json.dumps(health_2, indent=2, sort_keys=True))
print("PASS: duplicate evidence covers repeated content and cross-split leakage.")

**Reflection bridge:** Group-aware splitting protects evaluation, but says nothing about whether a row can teach the intended target. Two structurally valid SFT rows are operationally unusable.

## 3 - Chat Template and Label Validity

A schema can restrict role names without understanding role order. SFT needs a user request, no assistant answer before that request, and a final assistant response. Preference rows keep chosen and rejected responses outside `messages`, so their prompt ends with the user.

```mermaid
flowchart LR
    A["Schema-valid messages"] --> B{"Task type"}
    B -->|"SFT"| C["optional system, user, assistant target"]
    B -->|"preference"| D["optional system, user plus pair"]
    C --> E{"Order and target valid?"}
    D --> F{"Prompt ends with user?"}
    E -->|"no"| G["Quarantine row"]
    F -->|"yes"| H["Continue"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Will schema validation catch either malformed SFT conversation? The next cell identifies the semantic invariant each violates.

In [ ]:
# -- Validate conversational order and target availability ------------------
def template_issues(row: dict) -> list[str]:
    roles = [message["role"] for message in row["messages"]]
    issues = []
    if "user" not in roles:
        issues.append("missing_user_turn")
    elif "assistant" in roles and roles.index("assistant") < roles.index("user"):
        issues.append("assistant_before_user")
    if row["task_type"] == "sft" and (not roles or roles[-1] != "assistant"):
        issues.append("missing_final_assistant_target")
    if row["task_type"] == "preference" and roles and roles[-1] != "user":
        issues.append("preference_prompt_must_end_with_user")
    return issues


template_findings = {row["example_id"]: template_issues(row) for row in ROWS if template_issues(row)}
assert set(template_findings) == {"td-005", "td-006"}
assert "assistant_before_user" in template_findings["td-005"]
assert "missing_final_assistant_target" in template_findings["td-006"]
print(f"Measured template-invalid rows: {len(template_findings)}")
print(json.dumps(template_findings, indent=2, sort_keys=True))
print("Prediction resolved: both rows pass schema and fail semantic template checks.")

**Your turn:** Repair `td-006` only in memory by adding an assistant target. Confirm the shared row stays unchanged.

In [ ]:
# -- Your turn: repair a copied row without mutating shared fixtures --------
repaired_td_006 = deepcopy(ROWS_BY_ID["td-006"])
MY_ASSISTANT_TARGET = "Aria opened the maintenance log."  # CHANGE THIS: keep a non-empty answer
repaired_td_006["messages"].append({"role": "assistant", "content": MY_ASSISTANT_TARGET})
repair_passed = template_issues(repaired_td_006) == []
source_unchanged = ROWS_BY_ID["td-006"]["messages"][-1]["role"] == "user"
assert source_unchanged
print(f"Copied-row check passed: {repair_passed}")
print(f"Original fixture object unchanged: {source_unchanged}")
print("PASS: remediation creates a reviewed candidate rather than inventing a silent target.")

**Common Pitfalls**

| | Pattern | Why it matters |
| --- | --- | --- |
| Wrong | Validate allowed role names only | Valid names can appear in impossible order |
| Right | Validate task-specific conversation invariants | The example contains a target for its objective |
| Wrong | Auto-invent missing answers | Repair silently changes supervision |
| Right | Quarantine or create a reviewed replacement with lineage | Human intent remains attributable |

**Quick Health Check:** verify two invalid IDs, reasons per row, and every other row passing the semantic check.

In [ ]:
# -- Health check: semantic template validity ------------------------------
health_3 = {
    "two_invalid_rows": len(template_findings) == 2,
    "assistant_before_user_found": "assistant_before_user" in template_findings.get("td-005", []),
    "missing_target_found": "missing_final_assistant_target" in template_findings.get("td-006", []),
    "all_other_rows_valid": all(not template_issues(row) for row in ROWS if row["example_id"] not in template_findings),
}
assert all(health_3.values()), health_3
print(json.dumps(health_3, indent=2, sort_keys=True))
print("PASS: semantic validity is separate from structural validity.")

**Reflection bridge:** A valid target can still be data Riverside has no right to train on, or text that exposes contact details. Template checks protect the objective, not people, contracts, or evaluation boundaries.

## 4 - PII, Provenance, Licensing, and Contamination

Riverside needs four distinct signals: potential PII, missing provenance, unapproved rights status, and an eval-reserved source placed in training. Collapsing these into one `bad_row` flag erases the remediation path.

The fixture uses fictional reserved-domain and `555` probes. Regex proves detector wiring here; it does not establish production DLP recall. Rights fields are governance signals, not legal advice.

```mermaid
flowchart LR
    A["Schema-valid row"] --> B["Content scan"]
    A --> C["Provenance check"]
    C --> D["Rights or licensing status"]
    C --> E["Split-policy check"]
    B --> F{"PII?"}
    D --> G{"Approved?"}
    E --> H{"Eval source in train?"}
    F -->|"yes"| I["Redact or quarantine"]
    G -->|"no"| J["Approve or quarantine"]
    H -->|"yes"| K["Block contamination"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style J fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style K fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Unknown rights and split contamination both block promotion. Will the report preserve them as separate findings with separate owners?

In [ ]:
# -- Detect fictional PII probes and lineage-policy failures ----------------
PII_PATTERNS = {
    "email": re.compile(r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}", re.IGNORECASE),
    "fictional_555_phone": re.compile(r"\+1-\d{3}-555-\d{4}"),
}


def pii_findings(row: dict) -> list[dict]:
    text = learning_text(row)
    return [
        {"kind": kind, "match": match.group(0)}
        for kind, pattern in PII_PATTERNS.items()
        for match in pattern.finditer(text)
    ]


pii_by_id = {row["example_id"]: pii_findings(row) for row in ROWS if pii_findings(row)}
missing_provenance_ids = sorted(row["example_id"] for row in ROWS if row["provenance"] is None)
unapproved_rights_ids = sorted(
    row["example_id"] for row in ROWS
    if row["provenance"] is not None and row["provenance"]["rights_status"] != "synthetic_approved"
)
contamination_ids = sorted(
    row["example_id"] for row in ROWS
    if row["split"] == "train"
    and row["provenance"] is not None
    and row["provenance"]["split_policy"] == "eval_reserved"
)
provenance_present_count = sum(row["provenance"] is not None for row in ROWS)
approved_provenance_count = sum(
    row["provenance"] is not None and row["provenance"]["rights_status"] == "synthetic_approved"
    for row in ROWS
)

assert list(pii_by_id) == ["td-007"] and len(pii_by_id["td-007"]) == 2
assert missing_provenance_ids == ["td-008"]
assert unapproved_rights_ids == ["td-009"]
assert contamination_ids == ["td-013"]
assert provenance_present_count == 12 and approved_provenance_count == 11
print(f"Measured PII rows={len(pii_by_id)}, findings={sum(map(len, pii_by_id.values()))}")
print(f"Measured provenance present={provenance_present_count}/{len(ROWS)} ({provenance_present_count / len(ROWS):.1%})")
print(f"Measured approved provenance={approved_provenance_count}/{len(ROWS)} ({approved_provenance_count / len(ROWS):.1%})")
print(f"Missing provenance: {missing_provenance_ids}")
print(f"Unknown rights: {unapproved_rights_ids}")
print(f"Split-policy contamination: {contamination_ids}")
print("Prediction resolved: each governance failure remains separately attributable.")

**Your turn:** Add one narrow teaching pattern to a copied detector map and inspect every match. Do not infer production recall from this fixture.

In [ ]:
# -- Your turn: extend a detector without claiming production recall --------
MY_PATTERN_NAME = "reserved_test_email"  # CHANGE THIS: name the narrow probe
MY_PATTERN = re.compile(r"[A-Z0-9._%+-]+@example\.test", re.IGNORECASE)  # CHANGE THIS carefully
my_matches = {
    row["example_id"]: MY_PATTERN.findall(learning_text(row))
    for row in ROWS
    if MY_PATTERN.search(learning_text(row))
}
print(f"Detector: {MY_PATTERN_NAME}")
print(json.dumps(my_matches, indent=2, sort_keys=True))
print("PASS: matches are explicit; no recall claim is inferred.")

**Common Pitfalls**

| | Pattern | Why it matters |
| --- | --- | --- |
| Wrong | Store only `pii=true` | Reviewers cannot tell what matched or how to remediate |
| Right | Retain finding kind, match, row ID, detector version | Redaction review remains auditable |
| Wrong | Treat `source_uri` as permission | A location says where; rights status says whether |
| Right | Gate provenance, rights, and split policy separately | Each decision keeps an owner |

**Quick Health Check:** verify one PII row with two findings, one missing-lineage row, one unknown-rights row, one contaminated row, and full-dataset denominators.

In [ ]:
# -- Health check: safety, lineage, rights, and contamination ---------------
health_4 = {
    "one_pii_row": sorted(pii_by_id) == ["td-007"],
    "two_pii_findings": sum(map(len, pii_by_id.values())) == 2,
    "one_missing_provenance": missing_provenance_ids == ["td-008"],
    "one_unapproved_rights": unapproved_rights_ids == ["td-009"],
    "one_contaminated_train_row": contamination_ids == ["td-013"],
    "coverage_denominator_is_all_rows": provenance_present_count / len(ROWS) == 12 / 13,
}
assert all(health_4.values()), health_4
print(json.dumps(health_4, indent=2, sort_keys=True))
print("PASS: safety and lineage failures remain distinct evidence.")

**Reflection bridge:** Removing unsafe rows can make a dataset smaller and less representative. A gate must report what remains by task, split, and slice.

## 5 - Slice Balance and Coverage Risk

Thirteen rows are too few for training but enough to expose reporting failure. Riverside has four catalog rows and one security row. An aggregate count hides that four-to-one range and validation's single-slice coverage.

$$B=\frac{\min_s n_s}{\max_s n_s}$$

A lower $B$ means the thinnest slice is small relative to the largest. It is a diagnostic, not a universal threshold.

```mermaid
flowchart LR
    A["13 audited rows"] --> B["Count by task"]
    A --> C["Count by split"]
    A --> D["Count by slice"]
    D --> E["min/max ratio"]
    C --> F["Evaluation slice coverage"]
    E --> G{"Coverage warning?"}
    F --> G
    G -->|"yes"| H["Collect or qualify evidence"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Which slice is thinnest, and does validation cover it? The next cell computes both.

In [ ]:
# -- Measure task, split, slice, and validation coverage --------------------
task_counts = Counter(row["task_type"] for row in ROWS)
split_counts = Counter(row["split"] for row in ROWS)
slice_counts = Counter(row["slice"] for row in ROWS)
validation_slice_counts = Counter(row["slice"] for row in ROWS if row["split"] == "validation")
slice_balance_ratio = min(slice_counts.values()) / max(slice_counts.values())
assert task_counts == Counter({"sft": 10, "preference": 3})
assert split_counts == Counter({"train": 12, "validation": 1})
assert slice_counts == Counter({"catalog": 4, "finance": 3, "rights": 3, "editorial": 2, "security": 1})
assert validation_slice_counts == Counter({"rights": 1})
print(f"Measured task mix: {dict(sorted(task_counts.items()))}")
print(f"Measured split mix: {dict(sorted(split_counts.items()))}")
print(f"Measured slice mix: {dict(sorted(slice_counts.items()))}")
print(f"Measured min/max slice balance ratio: {slice_balance_ratio:.2f}")
print(f"Measured validation slices: {dict(sorted(validation_slice_counts.items()))}")
print("Prediction resolved: security is thinnest, and validation does not cover it.")

**Your turn:** Change the minimum row floor for a slice. Predict which slices will be flagged.

In [ ]:
# -- Your turn: apply a risk-specific slice floor ---------------------------
MY_MINIMUM_ROWS_PER_SLICE = 2  # CHANGE THIS: try 1, 3, or 4
thin_slices = sorted(name for name, count in slice_counts.items() if count < MY_MINIMUM_ROWS_PER_SLICE)
print(f"Minimum rows per slice: {MY_MINIMUM_ROWS_PER_SLICE}")
print(f"Slices below floor: {thin_slices}")
if thin_slices:
    print("Measured result: coverage needs collection or an explicit limitation.")
else:
    print("Measured result: this floor passes; sample adequacy still needs task evidence.")

**Common Pitfalls**

| | Pattern | Why it matters |
| --- | --- | --- |
| Wrong | Report only total rows | Dominant slices hide thin high-risk cases |
| Right | Report task, split, slice, and intersections | Coverage stays tied to evaluated behavior |
| Wrong | Force identical slice size | Equal counts can ignore different risks |
| Right | Set evidence floors from failure cost and traffic | Balance becomes a decision |

**Quick Health Check:** reconcile all totals, list every declared slice, compute the ratio from counts, and report validation separately.

In [ ]:
# -- Health check: distribution reconciliation -----------------------------
health_5 = {
    "task_total_reconciles": sum(task_counts.values()) == len(ROWS),
    "split_total_reconciles": sum(split_counts.values()) == len(ROWS),
    "slice_total_reconciles": sum(slice_counts.values()) == len(ROWS),
    "all_slices_present": set(slice_counts) == {"rights", "security", "finance", "editorial", "catalog"},
    "ratio_matches_counts": slice_balance_ratio == 1 / 4,
    "validation_is_separate": sum(validation_slice_counts.values()) == split_counts["validation"],
}
assert all(health_5.values()), health_5
print(json.dumps(health_5, indent=2, sort_keys=True))
print("PASS: aggregate totals and slice-level evidence reconcile.")

**Reflection bridge:** Slice counts show where evidence is thin. Preference data adds another risk: the same prompt can carry contradictory supervision, while a consistent label can reward a surface shortcut.

## 6 - Preference Disagreement and Shortcut Risk

Two preference rows share a comparison group and reverse chosen and rejected labels. A third chooses a much longer answer over a concise equivalent. The first is measurable disagreement; the second is a shortcut risk requiring review, not proof that length caused the label.

$$L=\frac{\operatorname{words}(chosen)}{\max(1,\operatorname{words}(rejected))}$$

A high $L$ is a review trigger. It does not prove semantic equivalence or annotator intent.

```mermaid
flowchart LR
    A["Preference rows"] --> B["Group by comparison ID"]
    B --> C{"Labels reverse?"}
    A --> D["Chosen/rejected word ratio"]
    D --> E{"Above threshold?"}
    C -->|"yes"| F["Unresolved disagreement"]
    E -->|"yes"| G["Shortcut-risk review"]
    F --> H["Block promotion"]
    G --> I["Warn and inspect semantics"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Does the length-risk row belong to the contradictory group? Keeping the findings separate preserves the correct remediation.

In [ ]:
# -- Measure contradictory labels and length shortcut risk ------------------
def word_count(text: str) -> int:
    return len(re.findall(r"[A-Za-z0-9']+", text))


def disagreement_groups(rows: list[dict]) -> dict[str, list[str]]:
    grouped = defaultdict(list)
    for row in rows:
        if row["task_type"] == "preference":
            grouped[row["comparison_group_id"]].append(row)
    disagreements = {}
    for group_id, group_rows in grouped.items():
        reversed_label = any(
            left["chosen"] == right["rejected"] and left["rejected"] == right["chosen"]
            for left, right in combinations(group_rows, 2)
        )
        if reversed_label:
            disagreements[group_id] = sorted(row["example_id"] for row in group_rows)
    return disagreements


def length_shortcut_rows(rows: list[dict], threshold: float) -> list[dict]:
    findings = []
    for row in rows:
        if row["task_type"] != "preference":
            continue
        chosen_words = word_count(row["chosen"])
        rejected_words = word_count(row["rejected"])
        ratio = chosen_words / max(1, rejected_words)
        if ratio >= threshold:
            findings.append({
                "example_id": row["example_id"],
                "chosen_words": chosen_words,
                "rejected_words": rejected_words,
                "length_ratio": round(ratio, 3),
            })
    return findings


LENGTH_SHORTCUT_REVIEW_THRESHOLD = 2.0
preference_disagreements = disagreement_groups(ROWS)
length_shortcuts = length_shortcut_rows(ROWS, LENGTH_SHORTCUT_REVIEW_THRESHOLD)
assert preference_disagreements == {"cg-001": ["td-010", "td-011"]}
assert [finding["example_id"] for finding in length_shortcuts] == ["td-012"]
print(f"Measured disagreement groups: {preference_disagreements}")
print(f"Measured length-shortcut review rows: {length_shortcuts}")
print("Prediction resolved: disagreement is cg-001; the separate length-risk row is td-012.")

**Your turn:** Change the length-ratio review threshold. Inspect measured ratios instead of treating the threshold as ground truth.

In [ ]:
# ── Your turn: test preference-review sensitivity ────────────────────────
MY_LENGTH_REVIEW_THRESHOLD = 2.0  # CHANGE THIS: try 1.25, then 3.0
my_length_reviews = length_shortcut_rows(ROWS, MY_LENGTH_REVIEW_THRESHOLD)
my_review_ids = [finding["example_id"] for finding in my_length_reviews]
expected_at_default = MY_LENGTH_REVIEW_THRESHOLD == LENGTH_SHORTCUT_REVIEW_THRESHOLD
default_check = (my_review_ids == ["td-012"]) if expected_at_default else True
print(f"Threshold {MY_LENGTH_REVIEW_THRESHOLD:.2f} flags: {my_review_ids}")
print(f"Default-threshold check: {'PASS' if default_check else 'FAIL'}")
print("Takeaway: the ratio prioritizes review; it does not prove label quality.")

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Treat a long chosen answer as proof of a bad label | Length is a proxy; semantics and annotator intent still need review |
| Wrong | Collapse disagreement and verbosity into one issue | The first needs adjudication; the second needs shortcut analysis |
| Right | Preserve comparison-group and row-level findings separately | Each issue reaches the owner who can resolve it |

**Quick Health Check:** confirm the reversed pair is exactly `td-010`/`td-011`, the length-risk row is `td-012`, and neither finding silently absorbs the other.

In [ ]:
# ── Health check: preference evidence stays attributable ────────────────
health_6 = {
    "one_disagreement_group": list(preference_disagreements) == ["cg-001"],
    "reversed_pair_exact": preference_disagreements.get("cg-001") == ["td-010", "td-011"],
    "length_risk_separate": [item["example_id"] for item in length_shortcuts] == ["td-012"],
    "no_overlap": "td-012" not in preference_disagreements.get("cg-001", []),
}
assert all(health_6.values()), health_6
for check, passed in health_6.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")
print("Takeaway: disagreement and shortcut risk remain distinct release evidence.")

**Reflection bridge:** Every defect now has an owner, but console findings disappear when the kernel stops. Riverside still cannot bind a training run to the exact source rows, curation decision, and release gate.

## 7 - Fingerprint the Candidate and Gate Promotion

A release gate needs two immutable identities: the raw source bytes and the canonical ordered candidate after reviewed curation. The notebook demonstrates quarantine as a reproducible policy, not as the only valid remediation. Production owners may redact, repair, relabel, adjudicate, or obtain rights instead.

```mermaid
flowchart LR
    A["Measured findings"] --> B["Issue ledger by row"]
    B --> C{"Blocking issue?"}
    C -->|yes| D["Quarantine for review"]
    C -->|no| E["Candidate row"]
    D --> F["Uncurated report: BLOCK"]
    E --> G["Canonical candidate digest"]
    G --> H["Curated report: conditional candidate"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** The fixture has 13 schema-valid rows. After every known issue is assigned, will the uncurated candidate pass, and how many rows remain in the conservative quarantine example: 0, 1, or 4?

In [ ]:
# ── Build the issue ledger, fingerprints, and promotion reports ─────────
def canonical_fingerprint(rows: list[dict]) -> str:
    ordered = sorted(rows, key=lambda row: row["example_id"])
    payload = json.dumps(ordered, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return sha256(payload.encode("utf-8")).hexdigest()


issue_ledger = defaultdict(list)
for cluster in cross_split_clusters:
    for example_id in cluster:
        issue_ledger[example_id].append("cross_split_leakage")
for example_id, reasons in template_findings.items():
    issue_ledger[example_id].extend(reasons)
for example_id in pii_by_id:
    issue_ledger[example_id].append("pii_detected")
for example_id in missing_provenance_ids:
    issue_ledger[example_id].append("missing_provenance")
for example_id in unapproved_rights_ids:
    issue_ledger[example_id].append("unapproved_rights")
for example_id in contamination_ids:
    issue_ledger[example_id].append("split_policy_contamination")
for group_rows in preference_disagreements.values():
    for example_id in group_rows:
        issue_ledger[example_id].append("preference_disagreement")
for finding in length_shortcuts:
    issue_ledger[finding["example_id"]].append("length_shortcut_review")

issue_ledger = {key: sorted(set(value)) for key, value in sorted(issue_ledger.items())}
blocked_ids = sorted(issue_ledger)
candidate_rows = [row for row in ROWS if row["example_id"] not in issue_ledger]
SOURCE_CANONICAL_SHA256 = canonical_fingerprint(ROWS)
CANDIDATE_SHA256 = canonical_fingerprint(candidate_rows)
uncurated_report = {
    "evidence_label": "LOCAL_FIXTURE when executed; UNVALIDATED in authored state",
    "source_file_sha256": SOURCE_FILE_SHA256,
    "source_canonical_sha256": SOURCE_CANONICAL_SHA256,
    "row_count": len(ROWS),
    "blocked_row_count": len(blocked_ids),
    "gate": "BLOCK" if blocked_ids else "PASS",
    "issues_by_example_id": issue_ledger,
}
curated_report = {
    "evidence_label": "LOCAL_FIXTURE when executed; UNVALIDATED in authored state",
    "policy": "quarantine every row with a known finding pending owner review",
    "candidate_sha256": CANDIDATE_SHA256,
    "candidate_row_count": len(candidate_rows),
    "candidate_ids": [row["example_id"] for row in candidate_rows],
    "gate": "CONDITIONAL_CANDIDATE" if candidate_rows else "BLOCK",
}
assert blocked_ids == [f"td-{index:03d}" for index in range(1, 14) if index != 4]
assert curated_report["candidate_ids"] == ["td-004"]
print(f"Measured issue coverage: {len(blocked_ids)}/{len(ROWS)} rows have a known finding")
print(f"Uncurated gate: {uncurated_report['gate']}")
print(f"Conservative candidate: {curated_report['candidate_row_count']} row, digest {CANDIDATE_SHA256[:12]}...")
print("Prediction resolved: the uncurated release is blocked; quarantine leaves only td-004.")

### Code Walkthrough: Release Evidence

1. **Canonical fingerprint** - rows are sorted by stable ID and serialized with fixed JSON separators, so row order in a copied file cannot silently change identity.
2. **Issue ledger** - every detector contributes a named reason to the affected row; no aggregate count discards ownership.
3. **Two reports** - the uncurated report records why training is blocked; the curated report records the exact quarantine policy and candidate digest.
4. **Honest label** - authored source is `UNVALIDATED`. Only a retained run over the named fixture may be labeled `LOCAL_FIXTURE`.

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Hash pretty-printed JSON in its current order | Formatting or order changes look like data changes |
| Wrong | Delete bad rows without a retained reason | The candidate cannot be reconstructed or reviewed |
| Right | Bind source digest, issue ledger, policy, candidate IDs, and candidate digest | The gate becomes reproducible release evidence |

**Quick Health Check:** write both reports, read them back, verify their digests and row counts, and confirm the shared fixture bytes did not change.

In [ ]:
# ── Health check: persist and re-read release evidence ──────────────────
CHAPTER_DIR = FIXTURE_DIR.parent.parent / "01-training-data-quality-and-lineage"
ARTIFACT_DIR = CHAPTER_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
report_paths = {
    "uncurated": ARTIFACT_DIR / "uncurated-data-quality-report.json",
    "candidate": ARTIFACT_DIR / "candidate-data-quality-report.json",
}
for name, report in (("uncurated", uncurated_report), ("candidate", curated_report)):
    report_paths[name].write_text(json.dumps(report, indent=2, sort_keys=True), encoding="utf-8")
reloaded_uncurated = json.loads(report_paths["uncurated"].read_text(encoding="utf-8"))
reloaded_candidate = json.loads(report_paths["candidate"].read_text(encoding="utf-8"))
health_7 = {
    "source_bytes_unchanged": sha256(DATA_PATH.read_bytes()).hexdigest() == SOURCE_FILE_SHA256,
    "blocked_report_round_trip": reloaded_uncurated == uncurated_report,
    "candidate_report_round_trip": reloaded_candidate == curated_report,
    "candidate_digest_recomputed": canonical_fingerprint(candidate_rows) == CANDIDATE_SHA256,
}
assert all(health_7.values()), health_7
for check, passed in health_7.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")
print(f"Evidence paths: {report_paths}")
print("Takeaway: the decision survives a kernel restart because its inputs and outputs are bound by digest.")

**Your turn:** Change one field only in an in-memory copy of `td-004`. The source-file digest must stay fixed while the candidate digest changes.

In [ ]:
# ── Your turn: prove candidate fingerprints are content-sensitive ───────
mutated_candidate = deepcopy(candidate_rows)
MY_REVIEW_NOTE = "reviewed for release"  # CHANGE THIS: use any non-empty note
mutated_candidate[0]["metadata"] = {"review_note": MY_REVIEW_NOTE}
mutated_digest = canonical_fingerprint(mutated_candidate)
digest_changed = mutated_digest != CANDIDATE_SHA256
source_stable = sha256(DATA_PATH.read_bytes()).hexdigest() == SOURCE_FILE_SHA256
assert digest_changed and source_stable
print(f"Candidate digest changed: {digest_changed}")
print(f"Shared source digest stayed fixed: {source_stable}")
print("Takeaway: a candidate mutation creates a new identity without rewriting source evidence.")

**Reflection bridge:** Riverside can now prove why the source candidate is blocked and which exact row survives the demonstration policy. That evidence is necessary, but one row is not a trainable or representative dataset. The correct next action is remediation and recollection, not training.

## 8 - Evidence Handoff and Completed Roadmap

```mermaid
flowchart LR
    A["Source fixture"] --> B["Measured issue ledger"]
    B --> C["Blocked uncurated report"]
    B --> D["Reviewed remediation"]
    D --> E["Candidate fingerprint"]
    E --> F["Prompt and release lineage chapters"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

The final scorecard below references measurements already produced in this notebook. It does not convert an unexecuted authored notebook into evidence.

In [ ]:
# ── Closing scorecard: assemble measured fixture evidence ───────────────
scorecard = [
    ("Schema-valid rows", schema_valid_count, len(ROWS)),
    ("Rows with known findings", len(blocked_ids), len(ROWS)),
    ("Cross-split leakage clusters", len(cross_split_clusters), 0),
    ("Template-invalid rows", len(template_findings), 0),
    ("PII rows", len(pii_by_id), 0),
    ("Unresolved preference groups", len(preference_disagreements), 0),
    ("Conservative candidate rows", len(candidate_rows), len(ROWS)),
]
print("Every number below comes from a proof cell in this notebook.")
for label, observed, target in scorecard:
    print(f"{label:<34} observed={observed:<3} target={target}")
recommendation = "BLOCK AND REMEDIATE" if uncurated_report["gate"] == "BLOCK" else "REVIEW FOR PROMOTION"
print(f"Riverside decision: {recommendation}")
print("Takeaway: 13/13 schema validity coexists with 12/13 rows requiring action.")

### Three-Tier Coverage Ledger

| Tier | Techniques | Evidence boundary |
|---|---|---|
| Built and measured | JSON Schema validation; exact duplicate hashing; token-set leakage clustering; chat-role/target checks; deterministic PII probes; provenance/rights/split checks; slice counts; preference disagreement; length-ratio review; canonical SHA-256 fingerprints; promotion reports | Local fixture workflow executed successfully in the unified FDE environment; notebook outputs were cleared |
| Explained and illustrated | Quarantine, redaction, relabeling, adjudication, and rights approval as remediation choices | The notebook implements quarantine only so each alternative remains an explicit owner decision |
| Named with a reason | Embedding-based semantic deduplication; production DLP; legal approval; inter-annotator statistics; warehouse-scale data observability | These need models, governed services, reviewers, legal authority, or production-scale data beyond this deterministic chapter |

If you find a technique named above that does not appear in the tier table, that is exactly the bug this section exists to catch.

### Completed Roadmap

| Step | Failure exposed | What the notebook now proves when run |
|---:|---|---|
| 1 | Parsing was mistaken for fitness | 13/13 rows are structurally valid, so later gates are necessary |
| 2 | Exact hashing missed evaluation leakage | One three-row train/validation cluster is measured |
| 3 | Schema-valid chats lacked valid targets | `td-005` and `td-006` fail separate semantic invariants |
| 4 | Valid text was unsafe or untraceable | PII, provenance, rights, and split findings retain separate owners |
| 5 | Aggregate count hid thin coverage | Security has one row and no validation coverage |
| 6 | Preference labels carried two risks | One reversed group and one separate length-risk row are measured |
| 7 | Console findings could not gate release | Source/candidate digests and two retained reports bind the decision |
| 8 | A clean-looking file invited premature training | The release is blocked; only `td-004` survives the demonstration quarantine policy |

### Key Takeaways

- Schema-valid is not training-ready.
- Split leakage is a content relationship, not a filename property.
- A detector finding names a review obligation, not universal truth.
- Dataset identity includes curation policy and canonical content.
- A blocked gate with traceable reasons is a successful safety outcome.

> **Next:** [Prompt Release and Experimentation](../02-prompt-release-and-experimentation/README.md) consumes the same discipline at the application-behavior boundary. [Release Registry and Lineage](../04-release-registry-and-lineage/README.md) later binds the promoted dataset fingerprint into a complete release graph.